Code modified from some defined functions  from
https://github.com/czbiohub-sf/comparison-RNAVelo/blob/main/method-agreement/method_comparison_df_code.ipynb

In [1]:
import pandas as pd
import scanpy as sc
import anndata
import scvelo as scv
import numpy as np
import os
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import matplotlib

In [2]:
import json
from typing import Dict
import pandas as pd
import torch
from pathlib import Path
from itertools import repeat
from collections import OrderedDict
from collections.abc import Mapping
from scvelo.core import l2_norm, prod_sum, sum

1.computing cell-cell similarity across different methods

In [ ]:

def cell_cell_cosinesim_velocitygraph_v2(adata1, adata2, key1, key2):
    
    A = adata1.uns[key1].copy()  
    B = adata2.uns[key2].copy()
    mycells = adata1.obs.index.tolist().copy() #
    
    A_normalized = scv.utils.get_transition_matrix(adata=adata1, vgraph=A) 
    B_normalized = scv.utils.get_transition_matrix(adata=adata2, vgraph=B)
    
    if A.shape != B.shape:
        raise ValueError("Input matrices must have the same shape.")

    n_rows = A.shape[0] 
    cosine_similarities = {}
    mysimilarities = []

    for i in range(n_rows):  
        row_A = A_normalized[i, :].toarray()[0] 
        row_B = B_normalized[i, :].toarray()[0]
        
        if np.isnan(row_A).any() or np.isnan(row_B).any():  
            mysimilarities.append(np.nan)
        else:
            dot_product = np.inner(row_A, row_B) 
            norm_A = np.linalg.norm(row_A)  
            norm_B = np.linalg.norm(row_B)

            mysimilarities.append(dot_product / (norm_A * norm_B)) 
    
    cosine_similarities['cosine_similiarities'] = mysimilarities
    cosine_similarities['cell_ID'] = mycells

    df = pd.DataFrame.from_dict(cosine_similarities) 
    return df

In [ ]:

def cell_cell_sim_table(adatalist=[], namelist=[]):
    n = len(namelist) 
    mydfs = []
    
    for i in range(n):  
        for j in range(i+1, n): 
            minidf = cell_cell_cosinesim_velocitygraph_v2(adatalist[i], adatalist[j], 'velocity_graph', 'velocity_graph')
            minidf = minidf.set_index('cell_ID') 
            minidf = minidf.rename(columns={"cosine_similiarities": namelist[i]+'_'+namelist[j]}) 
            mydfs.append(minidf) 
            
    bigdf = pd.concat(mydfs, axis=1) 
    
    return bigdf

In [ ]:
data_dir = '/data_path/velocity_cosistency_1st_batch/'
save_dir="/result_path/methods_agree_1st_batch/A1/csv/" 
datasets=['Pancreas','DentateGyrus','Erythroid_Maturation','HumanBoneMarrow','Intestinal_organoid','mouse_retina'] 
methods=['velocyto','scvelo-sto','scvelo-dyn','MultiVelo','veloAE','veloVI','VeloVAE','uniTvelo',,'Deepvelo2024','pyro-velocity','cell2fate','latentvelo'] #'Dynamo','cellDancer',
matrices = []
cell_ids = []
ind=np.arange(0,6) #
for i in ind:
    #i=0
    print(datasets[i])
    os.chdir(f'/data_path/velocity_cosistency_1st_batch/{datasets[i]}')
    g_method_dir="/data_path/velocity_cosistency_1st_batch/g_3_method/"
    adata_Deepvelo2024=sc.read_h5ad(f'{datasets[i]}_Deepvelo2024_consistency_score.h5ad')
    adata_scVelo_dynamic=sc.read_h5ad(f'{datasets[i]}_scVelo_dynamic_consistency_score.h5ad')
    adata_scVelo_stochastic=sc.read_h5ad(f'{datasets[i]}_scVelo_stochastic_consistency_score.h5ad')
    adata_veloVI=sc.read_h5ad(f'{datasets[i]}_veloVI_consistency_score.h5ad')
    adata_VeloVAE=sc.read_h5ad(f'{datasets[i]}_VeloVAE_consistency_score.h5ad')
    adata_veloAE=sc.read_h5ad(f'{datasets[i]}_veloAE_consistency_score.h5ad')
    #adata_Dynamo=sc.read_h5ad(f'{datasets[i]}_Dynamo_consistency_score.h5ad')
    adata_velocyto=sc.read_h5ad(f'{datasets[i]}_velocyto_consistency_score.h5ad')
    #adata_cellDancer=sc.read_h5ad(f'{datasets[i]}_cellDancer_consistency_score.h5ad')
    adata_uniTvelo=sc.read_h5ad(f'{datasets[i]}_uniTvelo_consistency_score.h5ad')
    adata_MultiVelo=sc.read_h5ad(f'{datasets[i]}_MultiVelo_consistency_score.h5ad')
    adata_pyro_velocity=sc.read_h5ad(g_method_dir+datasets[i]+"/"+f'{datasets[i]}_pyro-velocity_consistency_score.h5ad')
    adata_cell2fate=sc.read_h5ad(g_method_dir+datasets[i]+"/"+f'{datasets[i]}_cell2fate_consistency_score.h5ad')
    adata_latentvelo=sc.read_h5ad(g_method_dir+datasets[i]+"/"+f'{datasets[i]}_latentvelo_consistency_score.h5ad')
    ##vkey
    adata_VeloVAE.uns['velocity_graph']=adata_VeloVAE.uns['vae_velocity_graph']
    adata_veloAE.uns['velocity_graph']=adata_veloAE.uns['new_velocity_graph']
    adata_MultiVelo.uns['velocity_graph']=adata_MultiVelo.uns['velo_s_norm_graph']
    adata_pyro_velocity.uns['velocity_graph']=adata_pyro_velocity.uns['velocity_pyro_graph']
    adata_cell2fate.uns['velocity_graph']=adata_cell2fate.uns['Velocity_graph']
    adata_latentvelo.uns['velocity_graph']=adata_latentvelo.uns['spliced_velocity_graph']
    ##caculate
    ccDF = cell_cell_sim_table([adata_velocyto,adata_scVelo_stochastic, adata_scVelo_dynamic,adata_MultiVelo,adata_veloAE,adata_veloVI,adata_VeloVAE,
                                             adata_uniTvelo,adata_Deepvelo2024,adata_pyro_velocity,adata_cell2fate, adata_latentvelo], 
                        ['velocyto','scvelo-sto','scvelo-dyn','MultiVelo','veloAE','veloVI','VeloVAE','uniTvelo','Deepvelo2024','pyro-velocity','cell2fate','latentvelo']) 
    print(ccDF.shape)
    ccDF.to_csv(save_dir+f'{datasets[i]}_cosine_similarity.csv')



Pancreas
(3696, 78)
DentateGyrus
(2930, 78)
Erythroid_Maturation
(9815, 78)
HumanBoneMarrow
(5780, 78)
Intestinal_organoid
(3831, 78)
mouse_retina
(2726, 78)
